In [4]:
# STEP 1: Imports, Client Setup, and Tool Definition
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Define actual Python functions
def get_current_weather(location):
    # You could also use a live API here!
    return f"Weather in {location}: 28°C, sunny"

# Define the schema for the LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "stock_price",
            "description": "Get the current stock price for a given ticker symbol",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The stock ticker symbol, e.g. AAPL",
                    }
                },
                "required": ["ticker"],
            },
        }
    },
]    

# STEP 2: Ask the LLM (First Turn)
messages = [{"role": "user", "content": "What is the weather in London?"}]

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

msg = response.choices[0].message

# STEP 3: Handle Tool Call and Get Final Answer (Second Turn)
if msg.tool_calls:
    tool_call = msg.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    
    # Execute the actual function dynamically
    if tool_call.function.name == "get_current_weather":
        result = get_current_weather(args["location"])

        messages.append(msg)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

        final_response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages
        )
        print(final_response.choices[0].message.content)


The current weather in London is 28°C and sunny. Please note that this information may not be up-to-date as my knowledge cutoff is March 1, 2023. For the most recent weather forecast, I recommend checking a reliable weather website or app.


In [2]:
import json, os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# --- Your real tool functions ---
def get_current_weather(location):
    weather_info = {"New York": "Sunny, 25°C", "San Francisco": "Foggy, 15°C"}
    return weather_info.get(location, "Weather unknown")

def stock_price(ticker):
    stock_info = {"AAPL": "$150", "GOOGL": "$2800"}
    return stock_info.get(ticker, "Price unknown")

# Dispatch map
functions = {
    "get_current_weather": get_current_weather,
    "stock_price": stock_price,
}

# --- Tool schemas (used by the API) ---
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "stock_price",
            "description": "Get the current stock price for a given ticker symbol",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The stock ticker symbol, e.g. AAPL",
                    }
                },
                "required": ["ticker"],
            },
        }
    },
]    

# --- Agentic loop ---
messages = [{"role": "user", "content": "What is the current weather in New York?"}]
final = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant. When a tool returns data, "
                       "always report it directly and confidently. Never say you "
                       "lack real-time access — the tools provide that access."
        },
        *messages  # unpack your existing messages
    ]
)
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)
msg = response.choices[0].message

if msg.tool_calls:
    tool_call = msg.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    
    # Dynamic dispatch — no hardcoding
    result = functions[tool_call.function.name](**args)
    
    messages.append(msg)
    messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": str(result)})
    
    final = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages)
    print(final.choices[0].message.content)

I'm just an AI, I don't have access to real-time weather information. However, I can suggest some ways for you to find the current weather in New York:

1. Check online weather websites: You can visit websites like AccuWeather, Weather.com, or the National Weather Service (NWS) to get the current weather conditions in New York.
2. Use a mobile app: Download a weather app on your smartphone, such as Dark Sky or Weather Underground, to get real-time weather updates.
3. Tune into local news: Watch local news or listen to radio stations in New York to get the latest weather forecast.
4. Check social media: Follow weather accounts or local news outlets on social media platforms like Twitter or Facebook to stay updated on the current weather conditions.

Please note that the weather can change quickly, so it's always a good idea to check for updates frequently.
